# 2026-10-05 Lab data 2: epochs and segments

**Author:** Kyle Shannon · **Notebook:** lab-data-2-epochs-and-segments · **Created:** 2026-10-08T13:24:25-07:00

**Purpose:** Where the lab drew the lines inside a session: whether its timeStamps can be trusted, what the third segment is, how the file tags map to environment and condition, and what falls outside the stamps.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401  registers the SciencePlots styles

import axis_of_travel_rnn
from environments import list_environments, load_environment
from axis_of_travel_rnn.store import resolve_store

plt.style.use(["science", "notebook", "no-latex"])
rng = np.random.default_rng(0)
store = resolve_store(None)
FIGURES = Path("figures")

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"axis-of-travel-rnn {axis_of_travel_rnn.__version__}  commit {commit}  {date.today()}")
print(f"store: {store.root} ({store.source})")
print(f"environments: {', '.join(list_environments())}")

## Data

The lab's MATLAB files, read only, from the `LAB_DATA` folder in `configs/local.yaml`. Nothing is copied into the repository or the artifact store. The cells load them through `src/proj_helpers/lab_data_access.py` (the reader, the catalog and two ways to pick sessions) and nothing else. Every definition that encodes something we learned lives in the chapter that teaches it.

The plots are made from the lab's recordings, so run `pixi run nbstripout-install` once per clone: git then stores this notebook without its outputs.

`RUN_ALL = False` makes the cells that loop over sessions run on a sample of 12, the first three of each group. Set it to True for the full pass over all 109 sessions, which takes minutes on the network volume. The totals quoted in the text are for the full pass. Sessions named in the text always run.

In [ ]:
from collections import Counter

from proj_helpers.lab_data_access import catalog, find, lab_root, load_mat, pick_scope

RUN_ALL = False
root = lab_root()
sessions = catalog(root)
scope = pick_scope(sessions, RUN_ALL)
print(f"lab data: {root}")
print(f"{len(sessions)} sessions; the loops below cover {len(scope)}")

## Analysis

### 2.1 · Epochs: can the lab's timeStamps be trusted?

#### Why this was examined
An epoch (arena, track, variant) says where in a session the rat was, and every later statistic is split by it.
The boundaries live in one variable, `timeStamps`, which the analysis file only sometimes has and which was made
upstream of the labeling tool. The first spec said: use `timeStamps`, and fall back to the long tracking gaps
(both lights unfilled for more than 10 s) if they are missing or fail a check. Both halves were guesses. Nobody had
checked that the stamps are valid in every session, nor that the gaps mark the same boundaries. Task V3, steps 1
to 4.

#### The question and the goal
Are the stamps valid, how do they sit against the tracking gaps, and what should the converter do for the sessions
with no stamps, for tiny segments, and for the one session where stamps and tracking disagree? Goal: a rule for
where epochs come from that is simple, and that is shown to reproduce the lab's own segments where we can check.

#### What we found (V3)
- 103 sessions have `timeStamps`: 71 with 4 entries (every baseline session that has them) and 32 with 6 (every
  variant session that has them). 6 have none: 5 JL1 sessions and NS14_rec14. So segment 3 exists exactly when the
  file name carries a variant tag.
- The values are integers no larger than N and strictly increasing, so they are 1-based tracking rows, start then
  end per segment: segment 1 the arena (the lab calls it the platform), 2 the track, 3 the variant. None is out of
  order or outside the tracking.
- Every segment edge lies inside tracked time. Gaps lie between consecutive segments, but gaps cannot be assumed to
  mark segment boundaries: three variant sessions have extra long gaps inside segment 3, and six early NS14 sessions
  (rec2, 3, 5, 6, 8, 9) are one continuous tracked stretch with the arena and track marks only 25 to 621 frames
  apart.
- Arena absent (segment 1 only 2 to 4 frames): NS14_rec1, NS14_rec19, NS16_rec10. NS14_rec19 has no tracking for
  its first 18,231 frames.
- A fallback of "the N longest tracked stretches, in time order" (N = 2 for no variant tag, 3 for one) matches the
  lab's stamps in 88 of 103 stamped sessions. The 5 JL1 sessions with no stamps each have exactly 2 stretches, so
  the fallback gives both. NS14_rec14 has 4 stretches of 10 s or more where 3 are expected: unreliable.
- NS15_rec20 is the messy one: the arena stamp ends at row 27,339 inside a 57 s untracked gap, and 222 s of
  tracked time pass before the track stamp starts at 41,084. A gap-based replacement would be worse than the stamps.

#### What it changed
- The stamps are authoritative when present and in order (103 of 103 are). How they line up with the tracking gaps
  is a warn that reports margins and any edge inside a gap.
- A segment shorter than 10 s (600 frames) is not an epoch; it is reported as "arena absent".
- The fallback applies only when the stamps are absent, and only if the number of tracked stretches of 10 s or more
  equals the expected 2 or 3. That gives the 5 JL1 sessions epochs and NS14_rec14 none.
- A session with no epochs is still written, with epoch and phase blank and a warn; `runs.epoch` is nullable.

#### Where it lives
Spec: Behavior, the epochs bullet. Plan: Checks, the warns about stamps. Dictionary: 1.2 (`timeStamps`) and 1.4.
Converter: `build_epochs` (task S8).

#### How to run it
`RUN_ALL = False` runs the cells that loop over sessions on a sample of 12 so you can read the notebook quickly.
Set it to True for the full pass over 109 sessions, which takes some minutes on the network volume. The totals
quoted below are for the full pass.

#### 0. The definitions this chapter rests on
- `DT` is the seconds per frame as the lab stored it (chapter 1.2): the tracking time is exactly frame * 0.016667.
- A tracking GAP is both lights unfilled for more than `MIN_SEGMENT` = 600 frames (10 s). A TRACKED STRETCH is what
  lies between gaps. `stretches` returns them as (first frame, last frame), 1-based and inclusive; `holding` says
  which stretch a frame is in, or None when it lies inside a gap.
- `tags_of` pulls the variant tag out of an analysis file name (everything except `tt` and `rmaps`), and
  `expected_segments` is how many stamped segments a session should have: 3 when the name carries a variant tag, 2
  when it does not. That rule is one of the findings above.

In [ ]:
DT = 0.016667
MIN_SEGMENT = 600


def tags_of(pattern):
    """The tags in an analysis file name: everything except 'tt' and 'rmaps', in lower case."""
    return [t.lower() for t in pattern.split("_") if t.lower() not in ("tt", "rmaps")]


def expected_segments(pattern):
    """How many timeStamps segments a session should have: 3 when the name carries a variant tag, else 2."""
    return 3 if tags_of(pattern) else 2


def stretches(unfilled):
    """The tracked stretches as (first frame, last frame): what lies between gaps of both lights unfilled over 10 s."""
    lost = unfilled.all(axis=1)                         # both lights unfilled
    edges = np.flatnonzero(np.diff(np.r_[0, lost.astype(int), 0]))
    gaps = [(int(a) + 1, int(b)) for a, b in zip(edges[::2], edges[1::2]) if b - a > MIN_SEGMENT]
    cuts = [(0, 0)] + gaps + [(len(lost) + 1, len(lost) + 1)]   # sentinels close the first and last stretch
    return [(g[1] + 1, h[0] - 1) for g, h in zip(cuts, cuts[1:]) if h[0] > g[1] + 1]


def holding(frame, spans):
    """The index of the span that holds a frame, or None when the frame lies in a gap."""
    return next((i for i, (a, b) in enumerate(spans) if a <= frame <= b), None)

#### 1. One session per name pattern: the stamps next to the tracking gaps
Reading by hand, as in the first pass. The stamps are 1-based rows. A gap is both lights unfilled for more than
600 frames, written as (first frame, last frame). Things to see: baseline patterns have 4 numbers, every variant
pattern has 6; a stamp edge sits a few seconds inside a stretch of tracking; gaps fall between segments.

In [ ]:
firsts = {}
for sid, s in sessions.items():
    firsts.setdefault((s["group"], s["pattern"]), sid)

for (group, pattern), sid in sorted(firsts.items()):
    s = sessions[sid]
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    unfilled = load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool)
    segments = [] if stamps is None else np.atleast_1d(stamps).astype(int).reshape(-1, 2).tolist()
    lost = unfilled.all(axis=1)                         # both lights unfilled
    edges = np.flatnonzero(np.diff(np.r_[0, lost.astype(int), 0]))
    gaps = [(int(a) + 1, int(b)) for a, b in zip(edges[::2], edges[1::2]) if b - a > MIN_SEGMENT]
    print(f"{group[:3]} {sid[:-4]}  N={len(unfilled)}  segments {segments}")
    print(f"{'':4s}{pattern:25s} gaps over 10 s {gaps}")

#### 2. (New, for looking) Four sessions as timelines
Grey bars are tracked stretches (gaps of 10 s or more cut out), coloured bars are the lab's segments (blue arena,
orange track, green variant). Read them left to right:
- JL1_rec23: the clean case, two segments each inside a stretch.
- NS14_rec19: nothing is tracked for the first five minutes, and the "arena" is a few frames: arena absent.
- NS15_rec20: the arena stamp ends in a gap and there is a long tracked stretch no stamp covers.
- NS14_rec2: an early session, one continuous stretch, so the gap rule cannot see the arena/track boundary at all.

In [ ]:
def timeline(ax, sid):
    s = sessions[sid]
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    unfilled = load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool)
    minutes = DT / 60
    ax.broken_barh([(a * minutes, (b - a) * minutes) for a, b in stretches(unfilled)], (1.1, 0.8), facecolors="0.75")
    if stamps is not None:
        segs = np.atleast_1d(stamps).astype(int).reshape(-1, 2)
        ax.broken_barh([(a * minutes, (b - a) * minutes) for a, b in segs], (0.1, 0.8),
                       facecolors=["tab:blue", "tab:orange", "tab:green"][: len(segs)])
    ax.set_yticks([0.5, 1.5], ["stamps", "tracked"])
    ax.set_title(sid, fontsize=9)


ids = [find(sessions, k) for k in ("JL1_rec23", "NS14_rec19", "NS15_rec20", "NS14_rec2")]
fig, axes = plt.subplots(len(ids), 1, figsize=(10, 7), sharex=False)
for ax, sid in zip(axes, ids):
    timeline(ax, sid)
axes[-1].set_xlabel("minutes into the recording")
plt.tight_layout()
plt.show()

#### 3. The segment check on every session
For each stamped session: are the stamps increasing and inside the tracking, does each segment edge sit in tracked
time, does a segment span a long gap, is there a gap between consecutive segments? Full-pass totals from the
findings:
  103 stamped (71 with 4 entries, 32 with 6), 6 with none;
  3 first segments shorter than 10 s; 1 session with a segment-1 edge inside a gap;
  6 sessions with no long gap between consecutive segments; 8 segments that span a long gap;
  tracked stretches against real segments: 58 sessions 2 of 2, 26 sessions 3 of 3, then a tail of odd ones
  (6 with 1 of 2, 6 with 3 of 2, 4 with 4 of 3, 1 with 5 of 3).
Margins from a stretch edge to a segment edge: median about 32 frames.

In [ ]:
tally, start_margins, end_margins, odd = Counter(), [], [], []
for sid, s in scope.items():
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    if stamps is None:
        tally[f"{s['group'][:3]}: no timeStamps"] += 1
        continue
    flat = np.atleast_1d(stamps).astype(int)
    unfilled = load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool)
    spans = stretches(unfilled)
    tally[f"{len(flat)} entries"] += 1
    if not ((np.diff(flat) > 0).all() and flat[0] >= 1 and flat[-1] <= len(unfilled)):
        tally["not increasing, or outside the tracking"] += 1
    real, flagged = [], False
    for k, (a, b) in enumerate(flat.reshape(-1, 2).tolist(), 1):
        if b - a < MIN_SEGMENT:
            tally[f"segment {k} shorter than 10 s"] += 1
            flagged = True
            continue
        real.append((a, b))
        i, j = holding(a, spans), holding(b, spans)
        if i is None or j is None:
            tally[f"segment {k} has an edge inside a gap"] += 1
            flagged = True
            continue
        tally["segment spans a long gap"] += i != j
        start_margins.append(a - spans[i][0])
        end_margins.append(spans[j][1] - b)
    for (_, b), (a, _) in zip(real, real[1:]):
        if holding(b, spans) == holding(a, spans):
            tally["no gap between consecutive segments"] += 1
            flagged = True
    tally[f"{len(spans)} stretches for {len(real)} segments"] += 1
    if flagged:
        odd.append(sid)

for key, n in sorted(tally.items()):
    if n:
        print(f"  {n:4d}  {key}")
print("margin from a stretch edge to a segment edge, frames (min, median, max): starts",
      np.percentile(start_margins, [0, 50, 100]).astype(int).tolist(), "ends",
      np.percentile(end_margins, [0, 50, 100]).astype(int).tolist())
print("sessions behind a short segment, an edge inside a gap, or no gap between segments:", odd)

#### 4. Would the gap fallback have reproduced the lab's segments?
This is the test of the original spec's fallback. `fallback` takes the 2 or 3 longest tracked stretches. `iou` is
how much two spans overlap (1.0 is identical). Full pass: it matches in 88 of 103 stamped sessions. Read the 15 that
do not: they are the reason the fallback is not trusted when stamps exist.

In [ ]:
def fallback(spans, pattern):
    longest = sorted(spans, key=lambda ab: ab[0] - ab[1])[: expected_segments(pattern)]   # longest first
    return sorted(longest)                                                                # back in time order


def iou(p, q):
    shared = max(0, min(p[1], q[1]) - max(p[0], q[0]) + 1)
    return shared / ((p[1] - p[0] + 1) + (q[1] - q[0] + 1) - shared)


agree, differ, unstamped = 0, [], []
for sid, s in scope.items():
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    spans = stretches(load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool))
    guess = fallback(spans, s["pattern"])
    if stamps is None:
        unstamped.append((sid, s["pattern"], guess, spans))
        continue
    real = [tuple(p) for p in np.atleast_1d(stamps).astype(int).reshape(-1, 2).tolist() if p[1] - p[0] >= MIN_SEGMENT]
    scores = [round(iou(p, q), 2) for p, q in zip(real, guess)] if len(real) == len(guess) else None
    if scores and min(scores) >= 0.95:
        agree += 1
    else:
        differ.append((sid, s["pattern"], real, guess, scores))

print(f"the fallback matches the lab's segments in {agree} of {agree + len(differ)} stamped sessions")
for sid, pattern, real, guess, scores in differ:
    print(f"  {sid} {pattern}\n      lab {real}\n      fallback {guess}  overlap {scores}")
print("\nsessions with no timeStamps: the fallback's segments, and all tracked stretches")
for sid, pattern, guess, spans in unstamped:
    print(f"  {sid} {pattern}\n      fallback {guess}\n      stretches {spans}")

#### 5. (New) The rule we settled on, as code
Stamps if they are present and valid; else the tracked stretches of 10 s or more if their number matches what the
file name says to expect; else nothing. This is the logic the converter's `build_epochs` has. Full pass: 103 sessions
from `timeStamps`, 5 from tracked stretches (the JL1 ones), 1 with none (NS14_rec14).

In [ ]:
def choose_segments(stamps, unfilled, expected):
    n = len(unfilled)
    if stamps is not None:
        flat = np.atleast_1d(stamps).astype(int)
        if flat.size % 2 == 0:
            pairs = flat.reshape(-1, 2)
            if (len(pairs) == expected and pairs.min() >= 1 and pairs.max() <= n
                    and (pairs[:, 0] < pairs[:, 1]).all() and (pairs[1:, 0] > pairs[:-1, 1]).all()):
                return pairs.tolist(), "timeStamps"
    spans = [(a, b) for a, b in stretches(unfilled) if b - a + 1 >= MIN_SEGMENT]
    return (spans, "tracked stretches") if len(spans) == expected else ([], "none")


sources, absent = Counter(), []
for sid, s in scope.items():
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    unfilled = load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool)
    segments, source = choose_segments(stamps, unfilled, expected_segments(s["pattern"]))
    sources[source] += 1
    short = [k for k, (a, b) in enumerate(segments, 1) if b - a + 1 < MIN_SEGMENT]
    if short:
        absent.append((sid, short))
    if source != "timeStamps":
        print(f"  {sid} {s['pattern']}: {source}, {len(segments)} segments")
print(dict(sources))
print("sessions with a segment under 10 s, which is not an epoch (expect 3 in the full pass):", absent)

#### Things to try
- NS15_rec20 again: if you moved the track stamp back to the end of the gap (row 27,754), what would change? Chapter 2.3 shows the runs that sit between the two stamps.
- For the six early NS14 sessions, the arena and track marks are 25 to 621 frames apart in one stretch. Plot the
  rat's position around the arena stamp: can you see the arena-to-track transition?
- The 30 ms by which `t` runs long (chapter 1.2) adds up over a session. Does it matter for a 10 s threshold? For
  the frame-based epochs, why not?

### 2.2 · What is segment 3, and why are epochs named by role?

#### Why this was examined
In the first look at `timeStamps` (chapter 2.1) 32 sessions had a third segment, 12 to 22 minutes long, in every
session whose name carries a tag after or before `rmaps`: infusion tags (`bilatmusc`, `leftACSF`...), `dark`, and
rotation tags (`plus45`, `plusquarter`, plus their `turn` versions). The epoch vocabulary then in the spec was arena,
track, track_rot45, track_rot90, track_return. It had no name for segment 3 of an infusion or dark session, and
calling it `track` would collide with segment 2 on the (session, epoch) key. Nobody knew what `turn` meant (`plus45`
and `plus45turn` both occur), whether "return" epochs exist, or whether the tag really describes the maze the rat
was on. Task V3, steps 5 and 6.

#### The question and the goal
What is segment 3 physically: the same maze as segment 2, a rotated one, or something else? Goal: decide the epoch
names and the environment and condition each tag implies, from the rat's own movement and not from file names.

#### The method
A footprint is the set of 20-pixel cells of the camera frame that the front light visited for at least half a second
in a segment. The overlap (intersection over union) of two footprints says how much the rat used the same floor
space: 1.0 identical, 0 disjoint. The scale for "same maze" is given by splitting a track segment in half and
comparing the halves. The scale for "different layout" is given by the arena against the track.

#### What we found (V3 steps 5 and 6)
- Baseline scale: halves of the track overlap 0.68 / 0.81 / 0.95 (min / median / max, n = 71). Arena against track
  overlaps 0.07 / 0.11 / 0.15.
- Segment 2 against segment 3: infusion (12 sessions) 0.70 to 0.86 and dark (1 session) 0.76, the same as the
  halves of the track. Segment 3 is the same maze. All 19 stamped rotation sessions: 0.14 to 0.23, down at the arena
  against track level. Segment 3 is a different layout. The two groups do not overlap at all.
- The `turn` tags overlap 0.16 to 0.23, the same as the plain ones. `turn` does not mean a return to the standard
  orientation; it is a naming suffix. No session has a fourth segment, so `track_return` never occurs.
- Within each of NS14, NS15 and NS16 the rotation sessions form two clusters by their segment 3 footprint: overlap
  0.31 to 0.79 inside a cluster, 0.14 to 0.23 across. The 90 and 90t sessions share a cluster in all three rats. The
  45t sessions sit as far from the 90 cluster as plain 45 does, but no pair compares 45t with 45 directly.
- Footprints cannot say which cluster is 45 degrees and which is 90. That rests on the tag names and on stage 2.
- The gap between segment 2 and 3 is only 35 to 65 s, too short for an infusion. So the drug, or the dark, may apply
  to the whole file.

#### What it changed
- Epochs are named by their role in the recording: `arena`, `track`, `variant` (segment 3). Environment and
  condition carry the rest. `track_rot45`, `track_rot90`, `track_return` and the condition `rot_return` are gone.
- Tag lookup: `plus45` and `plus45turn` give the variant environment `triple_t_rot45` and condition `rot45`;
  `plusquarter`, `plusQuarter` and `plusquarterturn` give `triple_t_rot90` and `rot90`; infusion tags give the same
  maze with their own condition on every epoch (whole file); `dark` likewise with condition `dark`, which was added.
- Open, and recorded: the tag names are trusted for which cluster is 45 and which 90.

#### Where it lives
Dictionary: 2.6 (the epoch, environment, condition and tag tables) and 1.4. Spec: Behavior, the epochs bullet.
Converter: `epoch_labels`, `protocol_of` (task S8).

#### How to run it
The baseline scale loops over sessions, so it follows `RUN_ALL`. The variant sessions (32 of them) are always run in
full, since they are the point.

#### 0. One more definition
`lights` gives the front and back light positions in camera pixels with NaN wherever that light's unfilled FLAG is set; the flag decides, never the pixel value (chapter 1.2). `MIN_SEGMENT` and `tags_of` are from chapter 2.1.

In [ ]:
def lights(proc, unfilled):
    """The front and back light positions in camera pixels, NaN wherever that light is unfilled."""
    front, back = proc[:, 2:4].astype(float), proc[:, 4:6].astype(float)   # astype makes copies
    front[unfilled[:, 0]] = np.nan
    back[unfilled[:, 1]] = np.nan
    return front, back

#### 1. Footprints and their overlap
32 by 24 cells of 20 pixels cover the 640 by 480 frame. A cell counts if the front light stayed in it for at least
30 frames in total.

In [ ]:
def front_xy(s):
    d = load_mat(s["dvt"], variables=["processedDVT", "samplesUnfilled"])
    return lights(d["processedDVT"], d["samplesUnfilled"].astype(bool))[0]


def footprint(front, a, b):
    xy = front[a - 1:b]
    xy = xy[~np.isnan(xy).any(axis=1)]
    counts, _, _ = np.histogram2d(xy[:, 0], xy[:, 1], bins=[32, 24], range=[[0, 640], [0, 480]])
    return counts >= 30


def iou(p, q):
    return (p & q).sum() / (p | q).sum()


def stamp_pairs(s):
    stamps = load_mat(s["analysis"], variables=["timeStamps"]).get("timeStamps")
    return None if stamps is None else np.atleast_1d(stamps).astype(int).reshape(-1, 2).tolist()

#### 2. The scale: halves of the track, and the arena against the track
Baseline sessions have no segment 3. Expect (full pass) halves of the track 0.68 / 0.81 / 0.95 and arena against
track 0.07 / 0.11 / 0.15. These two numbers are the ruler for everything below: above 0.68 is "the same floor",
below about 0.25 is "somewhere else".

In [ ]:
rows = []
for sid, s in scope.items():
    pairs = stamp_pairs(s)
    if pairs is None or len(pairs) != 2:
        continue
    front = front_xy(s)
    a, b = pairs[1]
    mid = (a + b) // 2
    rows.append({"halves": iou(footprint(front, a, mid), footprint(front, mid + 1, b)),
                 "arena": iou(footprint(front, *pairs[0]), footprint(front, a, b))
                 if pairs[0][1] - pairs[0][0] >= MIN_SEGMENT else None})
for name, label in (("halves", "first half of the track vs second half"), ("arena", "arena vs track")):
    v = [r[name] for r in rows if r[name] is not None]
    print(f"  {label:40s} {np.min(v):.2f} / {np.median(v):.2f} / {np.max(v):.2f}  (n={len(v)})")

#### 3. Segment 2 against segment 3, for every variant session
Read the tag column against the overlap column. Infusion and dark sit with the halves of the track (about 0.7 and
above). Every `plus45`, `plusquarter` and `turn` session sits down at 0.14 to 0.23. There is no overlap between
the two groups of numbers, which is why segment 3 can be named from the tag.

In [ ]:
variant_sessions = {sid: s for sid, s in sessions.items() if tags_of(s["pattern"])}
seg23 = {}
for sid, s in variant_sessions.items():
    pairs = stamp_pairs(s)
    if pairs is None or len(pairs) != 3:
        continue
    front = front_xy(s)
    seg23[sid] = iou(footprint(front, *pairs[1]), footprint(front, *pairs[2]))
print(f"{len(seg23)} variant sessions with three stamped segments")
for sid, v in sorted(seg23.items(), key=lambda kv: (tags_of(sessions[kv[0]]["pattern"])[0], kv[0])):
    print(f"  {tags_of(sessions[sid]['pattern'])[0]:16s} {sid[:-4]:22s} segment 2 vs 3 overlap {v:.2f}")

#### 4. (New, for looking) Same maze, rotated maze
Red is where the rat went in segment 2, blue in segment 3, purple both. An infusion session fills one shape in
purple. A rotation session shows two shapes crossing: the track turned about the room.

In [ ]:
def overlay(ax, sid):
    s = sessions[sid]
    pairs = stamp_pairs(s)
    front = front_xy(s)
    rgb = np.zeros((24, 32, 3))
    rgb[..., 0] = footprint(front, *pairs[1]).T
    rgb[..., 2] = footprint(front, *pairs[2]).T
    ax.imshow(rgb, extent=[0, 640, 480, 0], interpolation="nearest")       # y down, like the camera
    ax.set_title(f"{sid[:-4]}\n{tags_of(s['pattern'])[0]}  overlap {seg23[sid]:.2f}", fontsize=9)


examples = [next(sid for sid in seg23 if tags_of(sessions[sid]["pattern"])[0] == tag)
            for tag in ("bilatmusc", "plus45", "plusquarter")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, sid in zip(axes, examples):
    overlay(ax, sid)
plt.tight_layout()
plt.show()

#### 5. Do the rotation tags map to two angles?
For each rat, the overlap of segment 3 between every pair of rotation sessions. A block of high numbers is a cluster
of sessions on the same layout. Expect two blocks per rat, 45 and 90, with the `turn` sessions inside the block of
their plain tag. The 45t sessions are only in NS14, so 45t against 45 is the one comparison the data cannot make.

In [ ]:
LABELS = {"plus45": "45", "plus45turn": "45t", "plusquarter": "90", "plusquarterturn": "90t"}
for group in ("NS14Data", "NS15Data", "NS16Data"):
    picks = []
    for sid, s in variant_sessions.items():
        tag = tags_of(s["pattern"])[0]
        pairs = stamp_pairs(s)
        if s["group"] != group or tag not in LABELS or pairs is None:
            continue
        picks.append((LABELS[tag], sid.split("_")[1], footprint(front_xy(s), *pairs[2])))
    picks.sort(key=lambda p: (p[0], p[1]))
    names = [f"{label}:{rec}" for label, rec, _ in picks]
    print(f"\n{group}: overlap of the cells visited in segment 3, session by session")
    print(" " * 10 + "".join(f"{n:>10s}" for n in names))
    for name, (_, _, fp) in zip(names, picks):
        print(f"{name:10s}" + "".join(f"{iou(fp, other):10.2f}" for _, _, other in picks))

#### 6. The tag lookup, as code
What each tag gives each epoch. Every pattern in the data is run through it. Check three things: a tagless session
has only arena and track, both baseline; an infusion or dark session carries its condition on every epoch; the
rotation tags change only the variant epoch.

In [ ]:
# tag, lower case -> (environment of the variant, condition, whole file under the condition)
TAGS = {
    "plus45": ("triple_t_rot45", "rot45", False), "plus45turn": ("triple_t_rot45", "rot45", False),
    "plusquarter": ("triple_t_rot90", "rot90", False), "plusquarterturn": ("triple_t_rot90", "rot90", False),
    "bilatmusc": ("triple_t", "musc_bilateral", True), "leftmusc": ("triple_t", "musc_left", True),
    "bilatacsf": ("triple_t", "acsf_bilateral", True), "leftacsf": ("triple_t", "acsf_left", True),
    "dark": ("triple_t", "dark", True),
}


def epoch_labels(tag):
    labels = {"arena": ("open_arena", "baseline"), "track": ("triple_t", "baseline")}
    if tag is None:
        return labels
    environment, condition, whole_file = TAGS.get(tag, ("triple_t", tag, True))   # an unknown tag is kept as a condition
    if whole_file:
        labels = {"arena": ("open_arena", condition), "track": ("triple_t", condition)}
    labels["variant"] = (environment, condition)
    return labels


for tag in [None] + sorted(TAGS):
    print(f"{str(tag):16s} {epoch_labels(tag)}")
print("\ntags seen in the data:", Counter(t for s in sessions.values() for t in tags_of(s["pattern"])))

#### Things to try
- Which of the 45 and 90 clusters is really 45 degrees? Fit the rotation between the segment 2 and segment 3
  footprints (or between the two clusters) and compare with 45 and 90. This is a stage 2 question, but the answer
  would confirm the tag lookup.
- Is the `dark` session's segment 3 behaviour different from segment 2 in speed? Chapter 1.3's speed code is a
  start.
- Infusion sessions: is the maze the same but the behaviour different? This is exactly what the project's baseline
  filter removes.

### 2.3 · What falls outside the stamps

#### Why this was examined
The all-session sweep left a loose end: 77 events lay outside every segment the stamps define. The spec said a run lies "inside its epoch", so a run with no epoch would break a fail check. Task V4, steps 8 and 9.

#### The question and the goal
Where do the outside events sit, and does anything need to change to accommodate them? Goal: decide what the converter does with a run that starts outside every epoch.

#### What we found
- 75 events lie outside every segment once the 2 row-0 placeholders are set aside. 74 of them are in one session, NS15_rec20_20150916_001; the other is in NS14_rec1_20141124_001 (a dirty run's END, 5 frames past the track stamp, on a run that starts inside the track).
- NS15_rec20: stamps [[1109, 27339], [41084, 144696]]. No events in the arena, 315 in the track, and 74 (about 37 runs) from row 27,959 to the track stamp at 41,084: a span of 13,745 frames (229 s) of tracked time between the two stamps. The arena stamp ends inside the tracking gap 24,334 to 27,754, and the first outside event follows that gap. So the track probably began near 27,754 and its stamp is late by about 13,300 frames. Whether the lab left those runs out of the neural analysis on purpose is not known.

#### What it changed
- A run's epoch is the epoch that holds its start row, and it is allowed to be blank. The 37 runs in NS15_rec20 keep their `run` number and phase `run` and have a blank epoch, with a warn naming the session. Frames between epochs have a blank epoch. `runs.epoch` is nullable.
- The stamps stay authoritative; we do not move NS15_rec20's track stamp. If the lab meant to exclude those runs, the blank epoch lets a later filter do so; if not, nothing was lost.

#### Where it lives
Spec: Behavior, the runs bullet (last sentences). Plan: Checks, Warn (runs with no epoch). Dictionary: 1.4 and 2.3. Converter: `build_runs` (the epoch lookup).

#### How to run it
The named sessions are always run. The scan over many sessions follows `RUN_ALL`.

#### 1. Events outside every segment
For a stamped session, which events (row above 0) lie outside all segments, and how far from the nearest segment edge
(negative before it, positive after). Full pass: 75 events; 0 within 2 frames, 2 within 3 to 10 frames (NS14_rec1's 5
and one in NS15_rec20), none within 11 to 100, and 73 farther than 100 frames, all in NS15_rec20.

In [ ]:
def outside_events(s):
    a = load_mat(s["analysis"], variables=["spiral_events", "timeStamps"])
    if "timeStamps" not in a:
        return None
    ev = np.asarray(a["spiral_events"], dtype=float).reshape(-1, 3)
    segs = np.atleast_1d(a["timeStamps"]).astype(int).reshape(-1, 2)
    inside = np.zeros(len(ev), bool)
    for x, y in segs:
        inside |= (ev[:, 0] >= x) & (ev[:, 0] <= y)
    return ev, segs, ev[~inside & (ev[:, 0] > 0)]


distances = []
for sid in dict.fromkeys([find(sessions, "NS15_rec20"), find(sessions, "NS14_rec1"), *scope]):
    found = outside_events(sessions[sid])
    if found is None:
        continue
    ev, segs, out = found
    if len(out):
        d = out[:, 0][:, None] - segs.ravel()[None, :]
        near = d[np.arange(len(d)), np.abs(d).argmin(axis=1)]
        distances.extend(near.tolist())
        print(f"{sid:26s} {len(out):3d} of {len(ev)} events outside; signed distance to the nearest edge "
              f"{near.min():.0f} to {near.max():.0f} frames")
d = np.abs(distances)
for name, lo, hi in (("0 to 2 frames", 0, 2), ("3 to 10", 3, 10), ("11 to 100", 11, 100), ("over 100", 101, np.inf)):
    print(f"  {name:14s} {int(((d >= lo) & (d <= hi)).sum())}")

#### 2. NS15_rec20 laid out
Grey: tracked stretches. Coloured: the two stamped segments (blue arena, orange track). Black ticks: events inside a
stamped segment; red ticks: the events outside. The red ticks sit in the stretch of tracked time between the two
coloured bars, right after the gap in which the arena stamp ends.

In [ ]:
sid = find(sessions, "NS15_rec20")
s = sessions[sid]
ev, segs, out = outside_events(s)
unfilled = load_mat(s["dvt"], variables=["samplesUnfilled"])["samplesUnfilled"].astype(bool)
lost = unfilled.all(axis=1)
edges = np.flatnonzero(np.diff(np.r_[0, lost.astype(int), 0]))
gaps = [(int(a) + 1, int(b)) for a, b in zip(edges[::2], edges[1::2]) if b - a > 600]
print(sid, "stamps", segs.tolist())
print("gaps over 10 s", gaps)
bins, counts = np.unique(out[:, 0] // 10000 * 10000, return_counts=True)
print("outside events per 10,000 frames (first frame: count):", {int(b): int(c) for b, c in zip(bins, counts)})
print("first 8 outside events (row, label):", out[:8, [0, 2]].astype(int).tolist())

minutes = DT / 60
fig, ax = plt.subplots(figsize=(11, 3))
ax.broken_barh([(a * minutes, (b - a) * minutes) for a, b in stretches(unfilled)], (1.1, 0.8), facecolors="0.8")
ax.broken_barh([(a * minutes, (b - a) * minutes) for a, b in segs], (0.1, 0.8), facecolors=["tab:blue", "tab:orange"])
inside_mask = ~np.isin(ev[:, 0], out[:, 0])
ax.plot(ev[inside_mask, 0] * minutes, np.full(inside_mask.sum(), 2.3), "|", color="k", ms=8)
ax.plot(out[:, 0] * minutes, np.full(len(out), 2.3), "|", color="tab:red", ms=8)
ax.set_yticks([0.5, 1.5, 2.3], ["stamps", "tracked", "events"])
ax.set_xlabel("minutes")
ax.set_title(sid)
plt.show()

#### 3. NS14_rec1: the 5-frame case
The arena stamp is 4 frames (so there is no arena epoch), the track runs 628 to 83,728, and the one outside event is
the end of a dirty run at row 83,733. The run STARTS inside the track, so its epoch is `track`; its last five frames
fall past the epoch's last frame. That is harmless, and the reason a run's epoch is decided by its start row.

In [ ]:
sid = find(sessions, "NS14_rec1")
ev, segs, out = outside_events(sessions[sid])
print(sid, "stamps", segs.tolist(), " outside events (row, time, label):")
print(out)

#### Things to try
- NS15_rec20: run the same plots with the track stamp moved to 27,754. Do the 37 runs now look like the rest of the track (path ids, durations)?

## Notes

Settled here:
- The stamps are authoritative when present; the tracking-gap fallback applies only when they are absent and the number of tracked stretches matches (chapter 2.1).
- Epochs are named by role, arena, track and variant. Infusion and dark put the whole file under their condition; the rotation tags change only the variant (chapter 2.2).
- A run can start outside every epoch (NS15_rec20); its epoch is blank and the stamps are left alone (chapter 2.3).

Open: which rotation cluster is 45 degrees and which is 90 rests on the tag names until stage 2 registers the track.

Next: Lab data 3, the runs.